# WSA_01 — Scenario Construction

**Purpose.** Build five controlled temperature scenarios from the proven operational IFB frames and validate model-development support.

> Run from the project repository. Outputs are generated only from the project data and frozen model artifacts.

In [1]:
# Import libraries
from pathlib import Path
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT/'src').exists() and PROJECT_ROOT!=PROJECT_ROOT.parent: 
    PROJECT_ROOT=PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path: 
    sys.path.insert(0,str(PROJECT_ROOT))

CONFIG = PROJECT_ROOT / 'configs' / 'weather_sensitivity.yaml'
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/weather_sensitivity.yaml')

In [3]:
# Import modules for weather_sensitivity
from src.ontario_peak_risk.weather_sensitivity.common import load_config, ensure_dirs

In [ ]:
cfg, project_root = load_config(CONFIG)
paths = ensure_dirs(cfg, project_root)
print("Project root:", project_root)

Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk


In [5]:
from src.ontario_peak_risk.inference_feature_builder.run_ifb import run_ifb
from src.ontario_peak_risk.weather_sensitivity.scenarios import build_scenario_frames, validate_only_temperature_changed
from src.ontario_peak_risk.weather_sensitivity.validation import temperature_reference, add_domain_status

In [6]:
origin = pd.Timestamp(cfg["analysis"]["forecast_origin"])
fsas = cfg["analysis"]["fsas"]
temp = cfg["analysis"]["temperature_feature"]
ifb = run_ifb(
    project_root / cfg["paths"]["ifb_config"],
    forecast_origin=origin,
    fsas=fsas,
    execute_models=False,
)
frames = ifb["feature_frames"]
print("IFB frames ready:", origin)

e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


IFB frames ready: 2026-03-25 23:00:00


In [7]:
# Build the model-development temperature reference.
# The historical feature dataset stores the source weather variable as
# `Temp (°C)`, while the frozen operational model frame exposes the same
# quantity as `origin__Temp (°C)`.
import yaml

feature_data = pd.read_parquet(project_root / cfg['paths']['feature_dataset'])

# Read the source temperature column from the proven IFB configuration
# instead of assuming it exists in weather_sensitivity.yaml.
ifb_config_path = project_root / cfg['paths']['ifb_config']
with ifb_config_path.open('r', encoding='utf-8') as handle:
    ifb_cfg = yaml.safe_load(handle)

source_temp = ifb_cfg['columns']['temperature']

if source_temp not in feature_data.columns:
    raise KeyError(
        f"Historical temperature column '{source_temp}' was not found in "
        f"{cfg['paths']['feature_dataset']}."
    )

q = tuple(cfg['analysis']['caution_quantiles'])
ref = temperature_reference(feature_data, feature=source_temp, q=q)

print('Historical temperature feature:', source_temp)
print('Operational model temperature feature:', temp)
print('Temperature support:', ref)


Historical temperature feature: Temp (°C)
Operational model temperature feature: origin__Temp (°C)
Temperature support: {'min': -21.8, 'q_low': -11.9, 'q_high': 29.4, 'max': 35.8, 'n': 262803}


In [8]:
# Validate that the operational IFB frames contain the frozen temperature feature
# and persist the historical temperature-domain reference.
missing_temp_frames = []
for task in ['rf', 'xgb']:
    for h in range(1, 25):
        if temp not in frames[task][h].columns:
            missing_temp_frames.append(f'{task}:h{h:02d}')

if missing_temp_frames:
    raise KeyError(
        f"Frozen temperature feature '{temp}' is missing from: "
        + ', '.join(missing_temp_frames)
    )

reference_path = paths['reports_dir'] / 'WSA_01_temperature_reference.json'
reference_path.write_text(json.dumps(ref, indent=2), encoding='utf-8')

print('Operational temperature feature validation: PASS')
print('Temperature reference saved to:', reference_path)


Operational temperature feature validation: PASS
Temperature reference saved to: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\weather_sensitivity\WSA_01_temperature_reference.json


In [9]:
scenario_frames = build_scenario_frames(frames, cfg["analysis"]["scenarios_c"], temp)
for task in ["rf", "xgb"]:
    meta = json.loads(
        (
            project_root
            / cfg["paths"]["artifacts_dir"]
            / cfg["models"][f"{task}_subdir"]
            / "metadata.json"
        ).read_text()
    )
    features = meta["numeric_features"] + meta["categorical_features"]
    for h in range(1, 25):
        validate_only_temperature_changed(
            frames[task][h], scenario_frames[task][h], features, temp
        )
        scenario_frames[task][h] = add_domain_status(
            scenario_frames[task][h], ref, temp
        )
print("Only-temperature-change validation: PASS")

Only-temperature-change validation: PASS


In [10]:
audit = pd.concat(
    [
        scenario_frames["rf"][h][
            [
                "fsa",
                "forecast_origin",
                "target_timestamp",
                "horizon",
                "scenario",
                "temperature_delta_c",
                "baseline_temperature_c",
                temp,
                "temperature_domain_status",
            ]
        ]
        for h in range(1, 25)
    ],
    ignore_index=True,
)
audit.to_csv(paths["outputs_dir"] / "WSA_01_temperature_scenarios.csv", index=False)
audit.to_parquet(
    paths["outputs_dir"] / "WSA_01_temperature_scenarios.parquet", index=False
)
print(audit["temperature_domain_status"].value_counts())
display(audit.head(15))
print("WSA_01 RESULT: COMPLETE")

temperature_domain_status
NORMAL    720
Name: count, dtype: int64


,fsa,forecast_origin,target_timestamp,horizon,scenario,temperature_delta_c,baseline_temperature_c,origin__Temp (°C),temperature_domain_status
0,L4T,2026-03-25 23:00:00,2026-03-26,1,-5C,-5.0,2.7,-2.3,NORMAL
1,M5R,2026-03-25 23:00:00,2026-03-26,1,-5C,-5.0,4.3,-0.7,NORMAL
2,M5S,2026-03-25 23:00:00,2026-03-26,1,-5C,-5.0,4.3,-0.7,NORMAL
3,M6G,2026-03-25 23:00:00,2026-03-26,1,-5C,-5.0,4.3,-0.7,NORMAL
4,M9R,2026-03-25 23:00:00,2026-03-26,1,-5C,-5.0,2.7,-2.3,NORMAL
5,M9W,2026-03-25 23:00:00,2026-03-26,1,-5C,-5.0,2.7,-2.3,NORMAL
6,L4T,2026-03-25 23:00:00,2026-03-26,1,-2.5C,-2.5,2.7,0.2,NORMAL
7,M5R,2026-03-25 23:00:00,2026-03-26,1,-2.5C,-2.5,4.3,1.8,NORMAL
8,M5S,2026-03-25 23:00:00,2026-03-26,1,-2.5C,-2.5,4.3,1.8,NORMAL
9,M6G,2026-03-25 23:00:00,2026-03-26,1,-2.5C,-2.5,4.3,1.8,NORMAL


WSA_01 RESULT: COMPLETE
